# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage
from lib.tooling import tool

print("imprts donr")

imprts donr


In [5]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
load_dotenv(".env")

os.environ.pop("OPENAI_BASE_URL", None)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not loaded"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not loaded"

print(" Env ready:", os.getenv("OPENAI_API_KEY")[:10])

 Env ready: sk-proj-Z6


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [21]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game


@tool

def retrieve_game(query: str) -> str:
    """
    semantic search: Finds most results in the vector DB
    args:
    -qyery: a question about game industry.


    You'll receive results as list. Each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...)
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """

    chroma_client =chromadb.PersistentClient(path="chromadb")
    embedding_fn=embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )

    collection =chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

    results = collection.query(query_texts=[query],n_results=3)
    documents = results ["documents"][0]
    return "\n\n".join (documents)

    print ("retrieve tool done")

#### Evaluate Retrieval Tool

In [13]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport (BaseModel):
    useful: bool
    description: str

@tool

def evaluate_retrieval (question: str, retrieved_docs: str) -> str:

        """
        Based on the user's question and on the list of retrieved documents, it will analyze the isability of the documents to respond to that question.
        args:
        - question: original question from user
        -retrieved_docs: retrieved documents most similar to the user query in the Vector Database
        The result includes:
        -useful: whether the documents are useful to answer the question
        -description: description about the evaluation result
        """

        llm= LLM(model="gpt-4o-mini", temperature=0.0)
        messages = [
            SystemMessage(content="Your task is to evaluate if the documents are enough to respond the query. Give a detailed explanation, so it's possible to take an action to accept it or not."),
        UserMessage(content=f"Question: {question}\n\nRetrieved Documents:\n{retrieved_docs}")
        ]

        response = llm.invoke(messages, response_format=EvaluationReport)
        return response.content
        
        print ("eval ret tool donr")

#### Game Web Search Tool

In [32]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool

def game_web_search (question: str) -> str:
    """
    Semantic Search: Finds mosts results in vecctor DB
    args:
    - question: a question about game industry.
    """

    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    response = tavily_client.search(question)
    results = response.get("results",[])
    output = ""

    for r in results:
        output+= f"Title: {r['title']}\n"
        output+= f"Title: {r['content']}\n\n"
    return output

    print ("game web srch done")


### Agent

In [33]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent = Agent(
    model_name="gpt-4o-mini",
    instructions= """" you are udaplay, an expert gaming assistant.
    You help users find information about video games including release dates, platforms, developers, publishers and descriptions. 

    To answer questions, follow these steps:

    1. Use retrivalval_game to search the vector DB
    2. Use evaluate_retrieval to check if the results are useful
    3. If not useful, use game_web_search to search the web
    4. Provide a clear and a structured answer to the user

    Always cite where the information came from (database or web search).""",
    tools=[retrieve_game, evaluate_retrieval, game_web_search]

    )

print ("agent ready")

agent ready


In [34]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions =[
"When Pokémon Gold and Silver was released?",
"Which one was the first 3D platformer Mario game?",
"Was Mortal Kombat X realeased for Playstation 5?"
]

for question in questions:
    print (f"\n{'='*60}")
    print (f"Question:{question}")
    print ('='*60)
    run = agent.invoke(question)
    final_state = run.get_final_state()
    print (final_state["messages"][-1].content)



Question:When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver was released in 1999 for the Game Boy Color. These games are notable for being the second generation of Pokémon games, introducing new regions, Pokémon, and gameplay mechanics. 

(Source: Vector Database)

Question:Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[State

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes